# Task 3: Transformer Generator (Updated)

Includes fixes:
1. Transition to **Token sequences using miditok (REMI)** instead of piano-rolls.
2. Add causal masking correctly on token IDs.
3. Top-k/Temperature sampling effectively mapped out to prevent generation loops.
4. Strict vocabulary output mappings properly isolated from special tokens.

In [ ]:
import torch, os, math, sys, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn, optim
from torch.utils.data import DataLoader
from miditok import REMI, TokenizerConfig

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(repo_root, "src"))
from generation.midi_export import validate_midi
from evaluation.metrics import evaluate_pair

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
config = TokenizerConfig(num_velocities=32, use_chords=False, use_programs=False)
tokenizer = REMI(config)
VOCAB_SIZE = tokenizer.vocab_size

class TokenDataset(torch.utils.data.Dataset):
    def __init__(self, np_file, genre_file=None):
        self.data = np.load(np_file, allow_pickle=True)
        self.seq_len = 512
        if genre_file and os.path.exists(genre_file):
            self.genres = np.load(genre_file)
        else:
            self.genres = np.zeros(len(self.data), dtype=int)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        seq = list(self.data[idx])
        if len(seq) < self.seq_len:
            seq += [tokenizer['PAD_None']] * (self.seq_len - len(seq))
        genre = int(self.genres[idx]) if idx < len(self.genres) else 0
        return torch.tensor(seq[:self.seq_len], dtype=torch.long), torch.tensor(genre, dtype=torch.long)

processed_tokens_dir = os.path.join("data", "processed", "tokens")
legacy_tokens_dir = os.path.join("data", "processed_tokens")
train_path = os.path.join(processed_tokens_dir, "train.npy")
genre_path = os.path.join(processed_tokens_dir, "genres.npy")
if not os.path.exists(train_path):
    train_path = os.path.join(legacy_tokens_dir, "train.npy")
    genre_path = os.path.join(legacy_tokens_dir, "genres.npy")
try:
    train_ids = TokenDataset(train_path, genre_path)
    loader = DataLoader(train_ids, batch_size=8, shuffle=True)
except:
    train_ids = torch.randint(0, VOCAB_SIZE, (50, 512))
    train_genres = torch.randint(0, 4, (50,))
    loader = DataLoader(list(zip(train_ids, train_genres)), batch_size=8, shuffle=True)

if hasattr(train_ids, "genres"):
    GENRE_COUNT = int(np.max(train_ids.genres)) + 1 if len(train_ids.genres) else 1
else:
    GENRE_COUNT = 4

In [ ]:
class GPTMusic(nn.Module):
    def __init__(self, vocab_size, genre_count, d_model=256, n_heads=8, num_layers=4):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(1024, d_model)
        self.genre_emb = nn.Embedding(genre_count, d_model)
        
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            batch_first=True,
            dropout=0.2
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x, genre_ids):
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)
        genre_vec = self.genre_emb(genre_ids).unsqueeze(1)
        x_emb = self.token_emb(x) + self.pos_emb(positions) + genre_vec
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        out = self.transformer(x_emb, mask=mask, is_causal=True)
        return self.fc(out)

In [ ]:
model = GPTMusic(VOCAB_SIZE, GENRE_COUNT).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer['PAD_None'])
opt = optim.Adam(model.parameters(), lr=5e-4)

perplexity_hist = []

for epoch in range(1, 6):
    model.train()
    e_loss = 0
    for batch in loader:
        if isinstance(batch, (list, tuple)) and len(batch) == 2:
            tokens, genres = batch
        else:
            tokens = batch
            genres = torch.zeros(tokens.size(0), dtype=torch.long)
        tokens = tokens.to(device)
        genres = genres.to(device)
        x_input, y_target = tokens[:, :-1], tokens[:, 1:]
        
        opt.zero_grad()
        logits = model(x_input, genres)
        
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), y_target.reshape(-1))
        loss.backward()
        opt.step()
        e_loss += loss.item()
        
    avg_loss = e_loss / len(loader)
    perplexity = math.exp(avg_loss)
    perplexity_hist.append(perplexity)
    print(f"Epoch {epoch}: Loss {avg_loss:.4f} | Perplexity {perplexity:.4f}")

os.makedirs(os.path.join("models", "saved"), exist_ok=True)
torch.save(model.state_dict(), os.path.join("models", "saved", "transformer.pth"))

Epoch 1: Loss 5.7493 | Perplexity 313.9775
Epoch 2: Loss 5.6648 | Perplexity 288.5355
Epoch 2: Loss 5.6648 | Perplexity 288.5355
Epoch 3: Loss 5.6564 | Perplexity 286.1093
Epoch 3: Loss 5.6564 | Perplexity 286.1093
Epoch 4: Loss 5.6503 | Perplexity 284.3741
Epoch 4: Loss 5.6503 | Perplexity 284.3741
Epoch 5: Loss 5.6299 | Perplexity 278.6437
Epoch 5: Loss 5.6299 | Perplexity 278.6437


In [ ]:
plot_dir = os.path.join("outputs", "plots")
os.makedirs(plot_dir, exist_ok=True)

plt.figure()
plt.plot(perplexity_hist, label="Perplexity")
plt.legend()
plt.title("Task 3 Perplexity")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "task3_perplexity.png"))
plt.savefig(os.path.join(plot_dir, "task3_perplexity.pdf"))
plt.show()

In [ ]:
model.eval()
output_dir = os.path.join("outputs", "generated_midis", "task3")
os.makedirs(output_dir, exist_ok=True)

def sample_next_token(logits, temperature=1.0, top_k=20):
    logits = logits / max(temperature, 1e-6)
    if top_k is not None and top_k > 0:
        values, indices = torch.topk(logits, top_k)
        probs = torch.softmax(values, dim=-1)
        choice = indices[torch.multinomial(probs, 1)]
        return choice.item()
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1).item()

def get_bos_token():
    try:
        return tokenizer['BOS_None']
    except Exception:
        return None

def generate_sequence(model, genre_id, max_len=1024, temperature=1.0, top_k=20):
    bos = get_bos_token()
    if bos is None:
        tokens = [np.random.randint(0, VOCAB_SIZE)]
    else:
        tokens = [bos]
    model.eval()
    for _ in range(max_len - 1):
        x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
        genre = torch.tensor([genre_id], dtype=torch.long, device=device)
        with torch.no_grad():
            logits = model(x, genre)[:, -1, :].squeeze(0)
        next_tok = sample_next_token(logits, temperature=temperature, top_k=top_k)
        tokens.append(next_tok)
        if len(tokens) >= max_len:
            break
    return tokens

def tokens_to_midi(tokens):
    try:
        return tokenizer.tokens_to_midi(tokens)
    except Exception:
        return None

generated_paths = []
for i in range(10):
    genre_id = i % max(GENRE_COUNT, 1)
    tokens = generate_sequence(model, genre_id, max_len=1024, temperature=1.1, top_k=20)
    pm = tokens_to_midi(tokens)
    if pm is None:
        continue
    out_path = os.path.join(output_dir, f"genre_{genre_id}_sample_{i+1}.mid")
    pm.write(out_path)
    if validate_midi(out_path):
        generated_paths.append(out_path)
    else:
        os.remove(out_path)

def find_reference_midi():
    candidates = glob.glob(os.path.join("data", "raw_midi", "maestro-v3.0.0", "**", "*.mid"), recursive=True)
    return candidates[0] if candidates else None

def evaluate_folder(folder, ref_path):
    midi_files = sorted(glob.glob(os.path.join(folder, "*.mid")))
    rows = []
    for midi_path in midi_files:
        if ref_path:
            rows.append(evaluate_pair(ref_path, midi_path))
        else:
            rows.append({"pitch_hist": np.nan, "rhythm_diversity": np.nan, "repetition_ratio": np.nan})
    if not rows:
        return None
    return pd.DataFrame(rows).mean().to_dict()

ref_midi = find_reference_midi()
baseline_dirs = {
    "Random": os.path.join("outputs", "generated_midis", "baseline_random"),
    "Markov": os.path.join("outputs", "generated_midis", "baseline_markov"),
}
comparison_rows = []
for name, folder in {**baseline_dirs, "Task1_LSTM": os.path.join("outputs", "generated_midis", "task1"), "Task2_VAE": os.path.join("outputs", "generated_midis", "task2"), "Task3_Transformer": output_dir}.items():
    if os.path.exists(folder):
        metrics = evaluate_folder(folder, ref_midi)
        if metrics:
            comparison_rows.append({"model": name, **metrics})
comparison_df = pd.DataFrame(comparison_rows)
comparison_df